# Basira — train a Kraken Arabic **handwriting** model (Muharaf)

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

Transfer-learns from the OpenITI Arabic base model on the `aamijar/muharaf-public`
line dataset (24,495 line image↔text pairs). Output: **`muharaf_hw_best.mlmodel`**.
~1–3 h on a free T4.

⚠️ **Licence:** Muharaf is **CC BY-NC-SA (non-commercial)** — use this model for
R&D / the stakeholder demo only. It installs into Basira's demo-gated slot.


In [ ]:
# 1) Install Kraken + check the GPU
!pip install -q kraken datasets pillow
import torch
print("CUDA available:", torch.cuda.is_available())   # must print True
# If pip says "restart": Runtime > Restart session, then re-run from here.


In [ ]:
# 2) Build Kraken training pairs (line image + .gt.txt) from the dataset
import os
from datasets import load_dataset

ds = load_dataset("aamijar/muharaf-public")           # public; no token needed
# QUICK TEST first? Uncomment to train on a small subset (~20 min):
# ds["train"] = ds["train"].select(range(2000))

for split in ds:                                       # train / validation / test
    out = f"data/{split}"; os.makedirs(out, exist_ok=True); n = 0
    for i, row in enumerate(ds[split]):
        text = (row.get("text") or "").strip()
        if not text:
            continue
        img = row["image"]
        if img.mode not in ("RGB", "L"):
            img = img.convert("RGB")
        img.save(f"{out}/{i:06d}.png")
        with open(f"{out}/{i:06d}.gt.txt", "w", encoding="utf-8") as fh:
            fh.write(text)
        n += 1
    print(split, "->", n, "line pairs")


In [ ]:
# 3) Fetch the OpenITI Arabic base model (for transfer learning)
!kraken get 10.5281/zenodo.7050270
import glob, os
base = glob.glob(os.path.expanduser("~/.local/share/htrmopo/**/*.mlmodel"), recursive=True)[0]
print("base model:", base)


In [ ]:
# 4) Train (transfer-learn). Writes muharaf_hw_best.mlmodel at peak val accuracy.
cmd = f'ketos train -d cuda:0 -f path --resize union -i "{base}" -o muharaf_hw data/train/*.png'
print(cmd)
!{cmd}
# If --resize union is rejected: run `!ketos train --help` and use the resize value shown.


In [ ]:
# 5) Evaluate on the held-out test split (character accuracy / CER)
!ketos test -d cuda:0 -f path -m muharaf_hw_best.mlmodel data/test/*.png


In [ ]:
# 6) Get the model off the runtime
import os
print("Model at:", os.path.abspath("muharaf_hw_best.mlmodel"))
# Browser Colab: triggers a download. In VS Code this may not work — then use the
# Drive option below, or download the file from the remote file explorer.
try:
    from google.colab import files
    files.download("muharaf_hw_best.mlmodel")
except Exception as e:
    print("files.download unavailable here:", e)
# Robust (works anywhere): save to Google Drive
# from google.colab import drive; drive.mount("/content/drive")
# !cp muharaf_hw_best.mlmodel /content/drive/MyDrive/


## Back on your Mac
With the app + Kraken sidecar running:
```bash
scripts/install-kraken-model.sh ~/Downloads/muharaf_hw_best.mlmodel muharaf
```
Then set `DEMO_TRANSCRIBE_ADAPTER=kraken-muharaf` in `.env`, restart (`pnpm dev`),
log in as `demo@basira.test`, and transcribe — it runs your handwriting model.
